Modélisation des variables comportemental des clients

In [46]:
import pandas as pd
import numpy as np
from pathlib import Path

In [47]:
DATA_PATH = Path(r"C:\Users\rayan\Downloads\Datasets (1)\Datasets")

clients = pd.read_csv(DATA_PATH / "clients.csv")
qualifications = pd.read_csv(DATA_PATH / "qualifications.csv")
promesses = pd.read_csv(DATA_PATH / "promesses.csv")

conversion des date en datetime

In [48]:
qualifications["date_qualification"] = pd.to_datetime(
    qualifications["date_qualification"]
)

promesses["date_promesse"] = pd.to_datetime(
    promesses["date_promesse"]
)

creation du cible au sein de qualification

In [49]:
qualifications = qualifications.merge(
    promesses[["id_qualification"]],
    on="id_qualification",
    how="left",
    indicator=True
)

qualifications["promise"] = (
    qualifications["_merge"]=="both"
).astype(int)

qualifications.drop(columns="_merge", inplace=True)

qualifications.head()

,id_qualification,id_client,date_qualification,type,statut,promise
0,3273,94,2025-09-12,SMS,Joignable,0
1,12310,294,2024-06-21,Deplacement,Joignable,0
2,878,28,2024-07-24,Deplacement,Joignable,0
3,1490,29,2024-03-24,Deplacement,Joignable,0
4,2861,62,2025-12-18,Email,En cours,0


sort chronologique

In [50]:
qualifications["date_qualification"] = pd.to_datetime(
    qualifications["date_qualification"]
)

qualifications = qualifications.sort_values(
    ["id_client", "date_qualification"]
).reset_index(drop=True)

In [51]:
promesse_ids = set(promesses["id_qualification"])

qualifications["promise"] = (
    qualifications["id_qualification"]
    .isin(promesse_ids)
    .astype(int)
)

qualifications.head()

,id_qualification,id_client,date_qualification,type,statut,promise
0,388,1,2024-05-14,Appel,Injoignable,0
1,410,1,2024-06-23,Email,En cours,0
2,347,1,2024-06-26,Appel,Joignable,0
3,337,1,2024-07-24,Appel,En cours,0
4,348,1,2024-11-08,Appel,Injoignable,0


In [52]:
qualifications["previous_contacts"] = (

    qualifications

    .groupby("id_client")

    .cumcount()

)

qualifications[
    [
        "id_client",
        "id_qualification",
        "previous_contacts"
    ]
].head(20)

,id_client,id_qualification,previous_contacts
0,1,388,0
1,1,410,1
2,1,347,2
3,1,337,3
4,1,348,4
5,1,352,5
6,1,206,6
7,1,161,7
8,1,393,8
9,1,197,9


In [53]:
qualifications["previous_promises"] = (

    qualifications

    .groupby("id_client")["promise"]

    .transform(

        lambda x: x.shift(1).fillna(0).cumsum()

    )

    .astype(int)

)

In [54]:
qualifications["previous_promise_rate"] = np.where(

    qualifications["previous_contacts"] == 0,

    0,

    qualifications["previous_promises"]

    /

    qualifications["previous_contacts"]

)

In [55]:
qualifications["had_previous_promise"] = (

    qualifications["previous_promises"] > 0

).astype(int)

In [56]:
qualifications["previous_contact_date"] = (

    qualifications

    .groupby("id_client")["date_qualification"]

    .shift(1)

)

qualifications["days_since_previous_contact"] = (

    qualifications["date_qualification"]

    -

    qualifications["previous_contact_date"]

).dt.days

qualifications["days_since_previous_contact"] = (

    qualifications["days_since_previous_contact"]

    .fillna(-1)

    .astype(int)

)

In [57]:
qualifications["average_days_between_contacts"] = (

    qualifications

    .groupby("id_client")["days_since_previous_contact"]

    .transform(

        lambda x: x.replace(-1, np.nan)

        .expanding()

        .mean()

    )

)

qualifications["average_days_between_contacts"] = (

    qualifications["average_days_between_contacts"]

    .fillna(-1)

)

In [58]:
client_history = qualifications.merge(

    clients,

    on="id_client",

    how="left"

)

In [59]:
client_history = client_history[

    [

        "id_qualification",

        "id_client",

        "nom",

        "prenom",

        "specialite",

        "adresse",

        "ville",

        "region",

        "id_gestionnaire",

        "previous_contacts",

        "previous_promises",

        "previous_promise_rate",

        "had_previous_promise",

        "average_days_between_contacts",

        "days_since_previous_contact"

    ]

]

client_history.head()

,id_qualification,id_client,nom,prenom,specialite,adresse,ville,region,id_gestionnaire,previous_contacts,previous_promises,previous_promise_rate,had_previous_promise,average_days_between_contacts,days_since_previous_contact
0,388,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,0,0,0.0,0,-1.000000,-1
1,410,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,1,0,0.0,0,40.000000,40
2,347,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,2,0,0.0,0,21.500000,3
3,337,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,3,0,0.0,0,23.666667,28
4,348,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,4,0,0.0,0,44.500000,107


In [60]:
print(client_history.shape)

display(client_history.head())

display(

    client_history[

        client_history["id_client"] ==

        client_history["id_client"].iloc[0]

    ]

)

(13115, 15)


,id_qualification,id_client,nom,prenom,specialite,adresse,ville,region,id_gestionnaire,previous_contacts,previous_promises,previous_promise_rate,had_previous_promise,average_days_between_contacts,days_since_previous_contact
0,388,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,0,0,0.0,0,-1.000000,-1
1,410,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,1,0,0.0,0,40.000000,40
2,347,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,2,0,0.0,0,21.500000,3
3,337,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,3,0,0.0,0,23.666667,28
4,348,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,4,0,0.0,0,44.500000,107


,id_qualification,id_client,nom,prenom,specialite,adresse,ville,region,id_gestionnaire,previous_contacts,previous_promises,previous_promise_rate,had_previous_promise,average_days_between_contacts,days_since_previous_contact
0,388,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,0,0,0.0,0,-1.000000,-1
1,410,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,1,0,0.0,0,40.000000,40
2,347,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,2,0,0.0,0,21.500000,3
3,337,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,3,0,0.0,0,23.666667,28
4,348,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,4,0,0.0,0,44.500000,107
5,352,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,5,0,0.0,0,49.800000,71
6,206,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,6,0,0.0,0,43.000000,9
7,161,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,7,0,0.0,0,45.285714,59
8,393,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,8,0,0.0,0,40.000000,3
9,197,1,Hernandez,Louis,Agriculteur,"138 Boulevard Anfa, Marrakech",Marrakech,Marrakech-Safi,1,9,0,0.0,0,44.555556,81


In [61]:
client_history.to_csv(

    DATA_PATH / "client_history_features.csv",

    index=False,

    encoding="utf-8-sig"

)

print("Saved successfully.")

Saved successfully.


SUMMARY :

In [62]:
client_history.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_qualification,13115.0,NaN,NaN,NaN,6558.0,3786.118725,1.0,3279.5,6558.0,9836.5,13115.0
id_client,13115.0,NaN,NaN,NaN,149.837286,89.382023,1.0,83.0,140.0,218.0,304.0
nom,13115,40,David,802,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prenom,13115,30,Marguerite,715,NaN,NaN,NaN,NaN,NaN,NaN,NaN
specialite,13115,24,Artisan,826,NaN,NaN,NaN,NaN,NaN,NaN,NaN
adresse,13115,303,"109 Avenue des FAR, Béni Mellal",119,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ville,13115,15,Safi,1907,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,13115,8,Marrakech-Safi,3230,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_gestionnaire,13115.0,NaN,NaN,NaN,37.781624,22.08618,1.0,22.0,35.0,55.0,75.0
previous_contacts,13115.0,NaN,NaN,NaN,30.322608,23.730626,0.0,11.0,25.0,46.0,118.0
